In [ ]:
from pathlib import Path
import nest_asyncio
from rail.utils import catalog_utils
from rail_svc import db, local_sync
nest_asyncio.apply()

In [ ]:
try:
    await db.close_db()
except:
    pass
db.init_db()

#catalog_utils.load_yaml('sandbox_catalogs.yaml')
try:
    local_sync.funcs.load_catalog_yaml('sandbox_catalogs.yaml')
except:
    pass

In [ ]:
model_dir = Path('pz/projects/sandbox/data')
estimates_dir = Path('pz/projects/sandbox/data')

In [ ]:
algos = {
    'knn':'rail.estimation.algos.k_nearneigh.KNearNeighEstimator',
    'fzboost':'rail.estimation.algos.flexzboost.FlexZBoostEstimator',
    'tpz':'rail.estimation.algos.tpz_lite.TPZliteEstimator',
    'cmnn':'rail.estimation.algos.cmnn.CMNNEstimator',
    'lephare':'rail.estimation.algos.lephare.LephareEstimator',
    'bpz':'rail.estimation.algos.bpz_lite.BPZliteEstimator',
    'gpz':'rail.estimation.algos.gpz.GPzEstimator',
    'dnf':'rail.estimation.algos.dnf.DNFEstimator',
}

In [ ]:
def safe_insert_catalog_tag(name):
    try:
        local_sync.catalog_tag.get_row_by_name(name)
    except:
        local_sync.catalog_tag.create_row(name=name)

In [ ]:
def safe_insert_dataset(name, path, catalog_tag_name, **kwargs):
    try:
        local_sync.dataset.get_row_by_name(name)
    except:
        local_sync.dataset.create_row(name=name, path=path, catalog_tag_name=catalog_tag_name, **kwargs)

In [ ]:
def safe_insert_algo(name, class_name):
    try:
        local_sync.algorithm.get_row_by_name(name)
    except:
        local_sync.algorithm.create_row(name=name, class_name=class_name)

In [ ]:
def safe_insert_model(name, path, algo_name, catalog_tag_name, **kwargs):
    try:
        local_sync.model.get_row_by_name(name)
    except:
        local_sync.model.create_row(name=name, path=path, algo_name=algo_name, catalog_tag_name=catalog_tag_name, **kwargs)

In [ ]:
def safe_insert_estimator(name, model_name, **kwargs):
    try:
        local_sync.estimator.get_row_by_name(name)
    except:
        local_sync.estimator.create_row(name=name, model_name=model_name, config=kwargs)

In [ ]:
def safe_insert_estimates(name, path, estimator_name, dataset_name, **kwargs):
    try:
        local_sync.estimates.get_row_by_name(name)
    except:
        local_sync.estimates.create_row(
            name=name, path=path, estimator_name=estimator_name, dataset_name=dataset_name, **kwargs
        )

In [ ]:
def safe_insert_matched_dataset(name, catalog_tag_name, component_dataset_names, path, n_objects):
    try:
        local_sync.dataset.get_row_by_name(name)
    except:
        local_sync.funcs.create_matched_dataset(
            name, catalog_tag_name, component_dataset_names=component_dataset_names, path=path, n_objects=n_objects,
        )

In [ ]:
for cat in ['roman', 'rubin']:
    safe_insert_catalog_tag(cat)

for ver in ['10yr', '1yr']:
    dataset_names = []
    for cat in ['roman', 'rubin']:
        dataset_name = f'sandbox_{cat}_{ver}'
        dataset_names.append(dataset_name)
        dataset_file = f'pz/data/test/sandbox_test_{cat}_{ver}.hdf5'
        catalog_tag_name = cat
        safe_insert_dataset(dataset_name, dataset_file, catalog_tag_name, n_objects=40000, is_collection=False, validate_file=True)

    matched_dataset_name = f'sandbox_roman_plus_rubin_{ver}'
    catalog_tag_name = 'roman_plus_rubin'
    dataset_file = f'pz/data/test/sandbox_test_{catalog_tag_name}_{ver}.hdf5'
    safe_insert_catalog_tag(catalog_tag_name)
    safe_insert_matched_dataset(matched_dataset_name, catalog_tag_name, dataset_names, dataset_file, 40000)

In [ ]:
for k, v in algos.items():
    algo = k
    safe_insert_algo(name=algo, class_name=v)    
    for cat in ['roman', 'rubin', 'roman_plus_rubin']:
        for ver in ['10yr', '1yr']:
            catalog_tag_name = cat
            model_name = f'sandbox_{algo}_{cat}_{ver}'
            dataset_name = f'sandbox_{cat}_{ver}'
            estimator_name = f'sandbox_{algo}_{cat}_{ver}'
            estimates_name = f'sandbox_{algo}_{cat}_{ver}'
            model_file = str(model_dir / f'flagship_gold_{cat}_{ver}' /f'model_inform_{algo}.pkl')
            qp_file = str(estimates_dir / f'flagship_gold_{cat}_{ver}' / f'output_estimate_{algo}.hdf5')
            safe_insert_algo(name=algo, class_name=v)
            if algo not in ['bpz', 'fzboost', 'gpz', 'knn']:
                continue
            safe_insert_model(model_name, model_file, algo, catalog_tag_name, validate_file=False)
            safe_insert_estimator(estimator_name, model_name)
            safe_insert_estimates(estimates_name, qp_file, estimator_name, dataset_name, validate_file=True)

In [ ]:
algos

In [ ]:
local_sync.dataset.get_rows()

In [ ]:
local_sync.catalog_band_assoc.get_rows()

In [ ]:
test = local_sync.funcs.estimate_pdf(1, 1, 4)

In [ ]:
test

In [ ]:
ret_vals = local_sync.funcs.get_dataset_and_estimates(1)

In [ ]:
ret_vals

In [ ]:
ret_vals = local_sync.funcs.get_data_and_estimates_data(1, 4)

In [ ]:
ret_vals

In [ ]:
local_sync.estimator.get_rows()